1. We will collect the job description
2. we will summarise the job description
3. we will collect the resume
4. we will extract information from the resume
5. we will compare the job description and resume to find a match

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("groq_api_key")

In [ ]:
from langchain.chains.summarize import load_summarize_chain
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts.prompt import PromptTemplate
from langchain_groq import ChatGroq

loader=TextLoader(r"C:\Users\debaj\Desktop\resume_check\ATS_groq\Data\jd.txt")
documents=loader.load()


template = """
    Write a concise and short summary of the following job description,
    Description :{text}

"""
prompt = PromptTemplate(input_variables=["text"],template=template)

llm=ChatGroq(model="llama-3.1-8b-instant")

chain=load_summarize_chain(llm,chain_type="stuff",verbose=True)
summary=chain.invoke(documents)
summary

In [ ]:
jd_summary=summary["output_text"]
print(jd_summary)

In [ ]:
import PyPDF2

def pdf_to_text(pdf_path,txt_path):
    try:
        with open(pdf_path,"rb") as pdf_file:
            pdf_reader=PyPDF2.PdfReader(pdf_file)
            text_content=""
            for page in pdf_reader.pages:
                text_content+=page.extract_text()

        with open(txt_path,"w",encoding="utf-8") as txt_file:
            txt_file.write(text_content)
        print(f"Successfully converted {pdf_path} to {txt_path}")
    except Exception as e:
        pass

In [ ]:
pdf_to_text(r"C:\Users\debaj\Desktop\resume_check\ATS_groq\Subhajyoti_Maity_resume_2024_9.pdf",
            r"C:\Users\debaj\Desktop\resume_check\ATS_groq\Data\output.txt")


In [ ]:
resume_loader=TextLoader(
    file_path=r"C:\Users\debaj\Desktop\resume_check\ATS_groq\Data\output.txt",
    encoding="utf-8"
)

resume_document=resume_loader.load()

template = """
    Write a concise and short summary of the following resume,
    Resume :{text}

"""

prompt = PromptTemplate(input_variables=["text"],template=template)

resume_chain=load_summarize_chain(llm,chain_type="stuff",prompt=prompt,verbose=True)
resume_output_summary=resume_chain.invoke(resume_document)
resume_output_summary

In [ ]:
resume_summary=resume_output_summary["output_text"]

In [ ]:
template = """
    You are a skilled ATS(Applicany Tracking System) scanner with a deep understanding of data science and other non IT fields and ATS Functionality. Your task is to evaluate the resume summary:{resume_summary} against the provided job description summary:{jd_summary}. Give me the percentage of match if the resume matches with the job description. First output should come as percentage and then keywords missing and final thought about the candidate.

"""

prompt = PromptTemplate(input_variables=["resume_summary","jd_summary"],template=template)

llm=ChatGroq(model="llama-3.1-8b-instant")

llm_chain = prompt | llm
response=llm_chain.invoke({"resume_summary":resume_summary,"jd_summary":jd_summary})
print(response.content)

In [ ]:
complete_prompt = prompt.format(resume_summary=resume_summary,jd_summary=jd_summary)
print(complete_prompt)

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt= ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=f"{complete_prompt}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | llm

ai_msg= chain.invoke({
    "messages":[
        HumanMessage(content="Give me the percentage of match if the resume matches with the job description. First output should come as percentage and then keywords missing and final thought about the candidate"),
        AIMessage(content=response.content),
        HumanMessage("Tell me more about him?")
    ]
})

print(ai_msg.content)